# Sherm Quanty — HMM Regime Detector: Test & Backtest

**Self-contained notebook** for Google Colab / Kaggle. Nothing to upload — everything (the 5-state Gaussian-HMM tactical regime engine, data loading, regime classification, and a regime-driven backtest) is embedded below.

Pipeline:
1. Install deps.
2. Load Nifty + India-VIX data (yfinance → synthetic fallback if offline / rate-limited).
3. Build the 9-feature matrix, fit a 5-state Gaussian HMM on the training window only.
4. Label states by a composite bullishness score → `H_BULL / L_BULL / SIDEWAYS / L_BEAR / H_BEAR`.
5. Visualise the regime overlay + confidence.
6. Backtest a simple regime-driven long/flat strategy vs buy-and-hold (in-sample & out-of-sample).

> This mirrors `regime_engine_tactical.py` from the repo, adapted to run on **daily** bars in a notebook so it works out-of-the-box on Colab/Kaggle (intraday 2h history is broker-gated). The machinery — scaler-on-train-only, 5-state full-covariance HMM, composite labelling, SIDEWAYS override, transition warnings — is identical.

In [ ]:
# 1. Dependencies (Colab/Kaggle usually have pandas/numpy/matplotlib/sklearn already)
%pip install -q hmmlearn yfinance

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)
np.random.seed(42)
print('imports ok')

## Config

Tune these and re-run. `TRAIN_FRACTION` fits the scaler + HMM on the first N% of bars only (no look-ahead); everything after is genuine out-of-sample.

In [ ]:
DATA_START              = '2010-01-01'
TRAIN_FRACTION          = 0.70    # scaler + HMM fit on the first 70% of bars
N_STATES                = 5
CONFIDENCE_THRESHOLD_L  = 0.50    # below this max-prob -> SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20    # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25    # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# 9 swing features (daily-bar version of the tactical engine's feature set)
MOM_1W, MOM_3W, MOM_5W = 5, 15, 25   # momentum lookbacks in trading days
VOL_WIN, VOL_FAST, VOL_SLOW = 10, 5, 20
SWING_WIN = 20

FEATURE_COLS = ['ret', 'mom_1w', 'mom_3w', 'mom_5w',
                'vol', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

## 2. Data loading — yfinance with synthetic fallback

yfinance can be rate-limited on shared Colab/Kaggle IPs. If the download fails or returns too little, we drop to a regime-blocked synthetic series calibrated to documented NSE history so the notebook always runs end-to-end.

In [ ]:
def _generate_synthetic(start=DATA_START):
    """Regime-blocked GBM + OU VIX, calibrated to documented NSE history."""
    rng = np.random.default_rng(42)
    dates = pd.bdate_range(start=start, end=pd.Timestamp.today().normalize())
    # (start, end, daily_mu, daily_sigma, vix_mean, vix_vol_factor)
    REGIMES = [
        ('2010-01-01', '2010-10-31',  0.0008, 0.0095, 18.0, 0.8),
        ('2010-11-01', '2011-12-31', -0.0003, 0.0135, 24.0, 1.2),
        ('2012-01-01', '2013-08-31',  0.0002, 0.0085, 17.0, 0.7),
        ('2013-09-01', '2014-04-30',  0.0003, 0.0090, 15.0, 0.7),
        ('2014-05-01', '2015-12-31',  0.0010, 0.0075, 14.0, 0.6),
        ('2016-01-01', '2016-02-28', -0.0007, 0.0120, 22.0, 1.1),
        ('2016-03-01', '2016-10-31',  0.0008, 0.0070, 13.0, 0.6),
        ('2016-11-01', '2016-12-31', -0.0008, 0.0130, 21.0, 1.0),
        ('2017-01-01', '2017-12-31',  0.0009, 0.0065, 12.0, 0.5),
        ('2018-01-01', '2019-03-31', -0.0002, 0.0100, 20.0, 0.9),
        ('2019-04-01', '2020-01-31',  0.0005, 0.0085, 14.5, 0.7),
        ('2020-02-01', '2020-03-23', -0.0055, 0.0250, 55.0, 3.0),
        ('2020-03-24', '2020-12-31',  0.0015, 0.0130, 22.0, 1.2),
        ('2021-01-01', '2021-10-31',  0.0010, 0.0072, 13.0, 0.6),
        ('2021-11-01', '2022-06-30', -0.0003, 0.0105, 22.0, 1.0),
        ('2022-07-01', '2023-12-31',  0.0007, 0.0082, 13.5, 0.7),
        ('2024-01-01', '2024-05-31',  0.0008, 0.0075, 13.0, 0.6),
        ('2024-06-01', '2026-12-31',  0.0005, 0.0090, 15.0, 0.8),
    ]
    rmap = {}
    for s, e, mu, sig, vm, vf in REGIMES:
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        for d in dates:
            if s <= d <= e:
                rmap[d] = (mu, sig, vm, vf)
    level, prev_vix = 5200.0, 16.0
    nvals, vvals = [], []
    for d in dates:
        mu, sig, vm, vf = rmap.get(d, (0.0004, 0.0090, 16.0, 0.8))
        level *= np.exp(rng.normal(mu, sig))
        prev_vix = max(prev_vix + 0.05 * (vm - prev_vix) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nvals.append(level); vvals.append(prev_vix)
    nifty = pd.Series(nvals, index=dates, name='nifty')
    vix   = pd.Series(vvals, index=dates, name='vix')
    return nifty, vix, True


def load_data(start=DATA_START):
    """Return (nifty_close, vix_close, is_synthetic)."""
    try:
        import yfinance as yf
        end = pd.Timestamp.today().strftime('%Y-%m-%d')
        nifty = yf.download('^NSEI', start=start, end=end, auto_adjust=True, progress=False)['Close'].squeeze()
        vix   = yf.download('^INDIAVIX', start=start, end=end, auto_adjust=True, progress=False)['Close'].squeeze()
        nifty.index = pd.to_datetime(nifty.index).normalize()
        vix.index   = pd.to_datetime(vix.index).normalize()
        nifty, vix = nifty.dropna(), vix.dropna()
        if len(nifty) > 500 and len(vix) > 500:
            aligned = pd.concat([nifty.rename('nifty'), vix.rename('vix')], axis=1, join='inner').dropna()
            print(f'yfinance OK -> {len(aligned)} aligned rows '
                  f'({aligned.index[0].date()} -> {aligned.index[-1].date()})')
            return aligned['nifty'], aligned['vix'], False
        raise ValueError('yfinance returned too little data')
    except Exception as e:
        print(f'yfinance unavailable ({e}); using synthetic data.')
        return _generate_synthetic(start)


nifty_close, vix_close, IS_SYNTHETIC = load_data()
print('synthetic:', IS_SYNTHETIC, '| rows:', len(nifty_close))
nifty_close.tail()

## 3. Feature engineering (9 swing features)

`StandardScaler` is fit on the **training slice only**, then applied to the whole series — so out-of-sample bars never leak into the fit.

In [ ]:
def build_features(nifty, vix, train_fraction=TRAIN_FRACTION):
    log_ret = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret']           = log_ret
    df['mom_1w']        = log_ret.rolling(MOM_1W).sum()
    df['mom_3w']        = log_ret.rolling(MOM_3W).sum()
    df['mom_5w']        = log_ret.rolling(MOM_5W).sum()
    df['vol']           = log_ret.rolling(VOL_WIN).std()
    df['vol_expansion'] = log_ret.rolling(VOL_FAST).std() / log_ret.rolling(VOL_SLOW).std()
    df['vix_chg']       = (vix - vix.shift(1)) / vix.shift(1)
    swing_high          = nifty.rolling(SWING_WIN).max()
    df['drawdown']      = (swing_high - nifty) / swing_high
    ma                  = nifty.rolling(SWING_WIN).mean()
    df['dist_ma']       = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    X_raw = df[FEATURE_COLS].values
    n_train = max(int(len(X_raw) * train_fraction), 50)
    scaler = StandardScaler().fit(X_raw[:n_train])
    X = scaler.transform(X_raw)
    return X, df.index, scaler, df, n_train


X, dates, scaler, feat_df, n_train = build_features(nifty_close, vix_close)
print(f'feature matrix: {X.shape} | train bars: {n_train} | oos bars: {len(X) - n_train}')
feat_df.tail()

## 4. Fit the 5-state HMM and label states

States are labelled by a **composite bullishness score** on their (z-scored) feature means, so an early momentum / vol expansion is recognised as H_BULL/H_BEAR sooner than a mean-return-only labelling would allow. A `SIDEWAYS` override kicks in when the leading-state probability is below `CONFIDENCE_THRESHOLD_L`.

In [ ]:
def composite_scores(model):
    m = model.means_
    ret, mom1, mom3, mom5, vol, volx, vixc, dd, distma = [m[:, i] for i in range(9)]
    return ret + 0.4 * (mom1 + mom3 + mom5) - vol - volx - vixc - dd + distma


def fit_and_classify(X, dates, nifty, vix, n_train):
    model = hmm.GaussianHMM(n_components=N_STATES, covariance_type='full',
                            n_iter=2000, random_state=42,
                            init_params='stmc', params='stmc')
    model.fit(X[:n_train])
    print('HMM converged:', model.monitor_.converged)

    order = np.argsort(composite_scores(model))
    state_labels = {order[0]: 'H_BEAR', order[1]: 'L_BEAR', order[2]: 'SIDEWAYS',
                    order[3]: 'L_BULL', order[4]: 'H_BULL'}

    states = model.predict(X)
    probs  = model.predict_proba(X)
    conf   = probs.max(axis=1)
    lead   = probs.argmax(axis=1)
    regime = np.array([
        'SIDEWAYS' if conf[i] < CONFIDENCE_THRESHOLD_L else state_labels[lead[i]]
        for i in range(len(dates))
    ])

    prob_cols = {lab: np.zeros(len(dates)) for lab in REGIME_LABELS}
    for raw, lab in state_labels.items():
        prob_cols[lab] = probs[:, raw]

    conf_s = pd.Series(conf, index=dates)
    vix_al = vix.reindex(dates)
    twf = ((conf_s.shift(1) - conf_s) > HMM_PROB_DROP_THRESHOLD) | \
          (((vix_al - vix_al.shift(1)) / vix_al.shift(1)) > VIX_SPIKE_THRESHOLD)
    twf.iloc[0] = False

    out = pd.DataFrame({
        'regime_state': regime,
        'regime_confidence': conf,
        'transition_warning_flag': twf.values.astype(bool),
        'hmm_state_int': states.astype(int),
        'nifty_close': nifty.reindex(dates).values,
        'vix_close': vix_al.values,
        **{f'prob_{lab}': prob_cols[lab] for lab in REGIME_LABELS},
    }, index=dates)
    out.index.name = 'date'
    return out, model, state_labels


regime_df, model, state_labels = fit_and_classify(X, dates, nifty_close, vix_close, n_train)

print('\nlearned state means (z-scored features):')
for raw, lab in sorted(state_labels.items(), key=lambda kv: kv[1]):
    print(f'  state {raw} -> {lab:<9} ret={model.means_[raw,0]:+.3f}  '
          f'mom5w={model.means_[raw,3]:+.3f}  vol={model.means_[raw,4]:+.3f}')

print('\nregime distribution:')
print(regime_df['regime_state'].value_counts().reindex(REGIME_LABELS).to_string())
print('\ntransition warnings:', int(regime_df['transition_warning_flag'].sum()))
regime_df.tail()

## 5. Visualise regime overlay + confidence

In [ ]:
def plot_regimes(df, split_date=None, synthetic=False):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 9),
                                   gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    suffix = '  [SYNTHETIC DATA]' if synthetic else ''
    fig.suptitle('Sherm Quanty — HMM Regime Detector' + suffix, fontsize=13)

    regimes, idx = df['regime_state'].values, df.index
    start_i = 0
    for i in range(1, len(regimes)):
        if regimes[i] != regimes[i - 1]:
            c = REGIME_COLORS.get(regimes[start_i], '#808080')
            ax1.axvspan(idx[start_i], idx[i - 1], alpha=0.35, color=c, lw=0)
            ax2.axvspan(idx[start_i], idx[i - 1], alpha=0.20, color=c, lw=0)
            start_i = i
    c = REGIME_COLORS.get(regimes[start_i], '#808080')
    ax1.axvspan(idx[start_i], idx[-1], alpha=0.35, color=c, lw=0)
    ax2.axvspan(idx[start_i], idx[-1], alpha=0.20, color=c, lw=0)

    ax1.plot(df.index, df['nifty_close'], color='black', lw=0.8)
    ax1.set_ylabel('Nifty 50')
    ax1.set_title('Price with regime background (5-state HMM)')

    ax2.plot(df.index, df['regime_confidence'], color='navy', lw=0.8)
    ax2.axhline(CONFIDENCE_THRESHOLD_L, color='orange', ls='--', lw=0.7)
    ax2.set_ylabel('confidence'); ax2.set_ylim(0, 1.05)

    for d in df.index[df['transition_warning_flag']]:
        ax1.axvline(d, color='purple', ls='--', lw=0.4, alpha=0.4)

    if split_date is not None:
        for ax in (ax1, ax2):
            ax.axvline(split_date, color='blue', ls='-', lw=1.2)
        ax1.text(split_date, ax1.get_ylim()[1], '  train | test', color='blue', va='top', fontsize=9)

    patches = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    ax1.legend(handles=patches, loc='upper left', ncol=5, fontsize=8)
    plt.tight_layout(); plt.show()


split_date = dates[n_train]
plot_regimes(regime_df, split_date=split_date, synthetic=IS_SYNTHETIC)

## 6. Backtest — regime-driven long/flat vs buy-and-hold

Simple rule: go **long** the index when the regime is bullish (`H_BULL` or `L_BULL`), else stay **flat**. Positions use the *previous* bar's regime to avoid look-ahead. We report the strategy vs buy-and-hold across the whole series and split out the out-of-sample slice.

In [ ]:
LONG_REGIMES = {'H_BULL', 'L_BULL'}

def backtest(df):
    d = df.copy()
    d['mkt_ret'] = d['nifty_close'].pct_change().fillna(0.0)
    d['position'] = d['regime_state'].isin(LONG_REGIMES).shift(1).fillna(False).astype(int)
    d['strat_ret'] = d['position'] * d['mkt_ret']
    d['bh_equity'] = (1 + d['mkt_ret']).cumprod()
    d['strat_equity'] = (1 + d['strat_ret']).cumprod()
    return d

def stats(returns, equity, periods_per_year=252):
    total = equity.iloc[-1] - 1
    years = len(returns) / periods_per_year
    cagr = equity.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    sharpe = (returns.mean() / returns.std() * np.sqrt(periods_per_year)) if returns.std() > 0 else np.nan
    dd = (equity / equity.cummax() - 1).min()
    return dict(total_return=total, cagr=cagr, sharpe=sharpe, max_drawdown=dd)

bt = backtest(regime_df)

def report(name, sub):
    s = stats(sub['strat_ret'], (1 + sub['strat_ret']).cumprod())
    b = stats(sub['mkt_ret'],  (1 + sub['mkt_ret']).cumprod())
    print(f'\n=== {name} ({sub.index[0].date()} -> {sub.index[-1].date()}, {len(sub)} bars) ===')
    print(f"{'metric':<16}{'regime long/flat':>18}{'buy & hold':>16}")
    for k in ('total_return', 'cagr', 'sharpe', 'max_drawdown'):
        fmt = (lambda v: f'{v:>17.2%}') if k != 'sharpe' else (lambda v: f'{v:>17.2f}')
        print(f'{k:<16}{fmt(s[k])} {fmt(b[k])}')
    exposure = sub['position'].mean()
    print(f"{'time in market':<16}{exposure:>17.1%} {1.0:>17.1%}")

report('FULL SAMPLE', bt)
report('OUT-OF-SAMPLE', bt.loc[split_date:])

In [ ]:
# Equity curves
fig, ax = plt.subplots(figsize=(20, 7))
ax.plot(bt.index, bt['bh_equity'],    label='Buy & Hold',       color='gray', lw=1.2)
ax.plot(bt.index, bt['strat_equity'], label='Regime long/flat', color='darkgreen', lw=1.4)
ax.axvline(split_date, color='blue', ls='-', lw=1.2)
ax.text(split_date, ax.get_ylim()[1], '  train | test', color='blue', va='top', fontsize=9)
ax.set_yscale('log'); ax.set_ylabel('growth of 1 (log)')
ax.set_title('Equity curve — regime-driven long/flat vs buy & hold' +
             ('  [SYNTHETIC DATA]' if IS_SYNTHETIC else ''))
ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

## 7. Inspect the latest regime + save results

`regime_df` holds the full classified history; write it out if you want to download it from Colab/Kaggle.

In [ ]:
latest = regime_df.iloc[-1]
print('latest bar :', regime_df.index[-1].date())
print('regime     :', latest['regime_state'])
print('confidence : {:.1%}'.format(latest['regime_confidence']))
print('warning    :', bool(latest['transition_warning_flag']))

regime_df.to_csv('regime_history.csv')
print('\nsaved -> regime_history.csv')

---
**Notes**
- If `synthetic: True` printed above, yfinance was blocked — results are illustrative only. Re-run later, or upload a `data/raw/NSEI_daily.csv` (columns `date,close`) and point `load_data` at it for real validation.
- Real-data sanity checks: 2014 & 2021 should skew `H_BULL/L_BULL`; Mar-2020 should show a sharp `H_BEAR`; 2022 should show `L_BEAR/H_BEAR`.
- The repo's production engine (`regime_engine_tactical.py`) runs this same machinery on **2h** bars via broker/yfinance intraday data; this notebook uses daily bars so it runs anywhere without a broker key.